# LeetCode 873 — Length of Longest Fibonacci Subsequence bằng Python

Link đề: https://leetcode.com/problems/length-of-longest-fibonacci-subsequence/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Dynamic Programming (quy hoạch động)

Skeleton LeetCode:

```python
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        ...
```

Ghi chú: `arr` **tăng ngặt**. Một dãy kiểu Fibonacci được **xác định hoàn toàn bởi hai phần tử đầu** (hoặc hai phần tử cuối của một đoạn đang xét), vì phần tử kế là tổng hai phần tử trước. Điều đó dẫn tới hai hướng: **chọn cặp đầu rồi kéo dài** (bằng `set`), hoặc **quy hoạch động theo cặp cuối** `(j, i)`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm dãy con (giữ thứ tự, bỏ được phần tử) dài nhất mà mỗi phần tử từ vị trí thứ 3 bằng tổng hai phần tử liền trước, độ dài tối thiểu 3; không có thì trả `0`.
2. Nắm các công cụ Python cần thiết: `set` và `dict` (giá trị -> chỉ số) để tra `O(1)`, `itertools.combinations`, vòng lặp kép trên cặp chỉ số, biểu thức "walrus" `:=` (tùy chọn), và mảng hai chiều `[[0] * n for _ in range(n)]`.
3. Hiểu nguyên lý: **trạng thái là cặp hai phần tử cuối** `(arr[j], arr[i])`; phần tử đứng trước chúng chắc chắn là `arr[i] - arr[j]` nên chỉ cần **một phép tra bảng băm**.
4. Đi qua 4 cách: vét cạn mọi dãy con, chọn cặp đầu rồi kéo dài bằng `set`, quy hoạch động hai chiều với bảng chỉ số (cách nộp chính), và quy hoạch động với **hai con trỏ** (không cần bảng băm); đối chiếu bằng hai oracle độc lập (liệt kê mọi tập con, và DFS tìm trong mảng theo vị trí).
5. Tránh các lỗi kinh điển: trả về `2` thay vì `0` khi không có dãy độ dài `>= 3`, quên điều kiện thứ tự `k < j`, chỉ thử cặp đầu tiên của mảng, và nhầm hướng xác định dãy.

## 2. Đề bài & ví dụ

Một dãy `x1, x2, ..., xn` được gọi là **kiểu Fibonacci** nếu:

- `n >= 3`;
- `xi + x(i+1) == x(i+2)` với mọi `i + 2 <= n`.

Cho mảng `arr` gồm các số nguyên dương **tăng ngặt**. Trả về **độ dài** của dãy con kiểu Fibonacci **dài nhất** của `arr`; nếu không có thì trả về `0`.

**Dãy con** nhận được từ `arr` bằng cách xóa một số phần tử (có thể không xóa) mà không đổi thứ tự các phần tử còn lại.

**Ví dụ 1**

- Input: `arr = [1,2,3,4,5,6,7,8]`
- Output: `5`
- Giải thích: dãy dài nhất `[1,2,3,5,8]`.

**Ví dụ 2**

- Input: `arr = [1,3,7,11,12,14,18]`
- Output: `3`
- Giải thích: `[1,11,12]`, `[3,11,14]` hoặc `[7,11,18]`.

**Ràng buộc:**

- `3 <= arr.length <= 1000`
- `1 <= arr[i] < arr[i + 1] <= 10^9`

In [1]:
from typing import List
from itertools import combinations
import random, time

examples = [
    ([1, 2, 3, 4, 5, 6, 7, 8], 5),
    ([1, 3, 7, 11, 12, 14, 18], 3),
]
for arr, expected in examples:
    print(f"arr={arr} kỳ vọng={expected}")

arr=[1, 2, 3, 4, 5, 6, 7, 8] kỳ vọng=5
arr=[1, 3, 7, 11, 12, 14, 18] kỳ vọng=3


## 3. Kiến thức nền

### 3.1. Dãy kiểu Fibonacci được xác định bởi hai phần tử

Nếu biết hai phần tử liên tiếp `x, y` thì phần tử kế là `x + y`, rồi `y + (x + y)`, ... Ngược lại, biết `y` và `z` (hai phần tử cuối) thì phần tử trước `y` là `z - y`. Do đó **hai phần tử liên tiếp bất kỳ** xác định cả đoạn trước và sau của dãy.

In [2]:
def extend(x: int, y: int, count: int) -> List[int]:
    seq = [x, y]
    for _ in range(count):
        seq.append(seq[-1] + seq[-2])
    return seq

print(extend(1, 2, 5))       # 1 2 3 5 8 13 21
print(extend(3, 11, 3))      # 3 11 14 25 39
print("phần tử trước (y=5, z=8):", 8 - 5)

[1, 2, 3, 5, 8, 13, 21]
[3, 11, 14, 25, 39]
phần tử trước (y=5, z=8): 3


### 3.2. `set`, `dict` và truy vấn "giá trị có trong mảng không, ở đâu"

Vì `arr` tăng ngặt nên các giá trị **khác nhau**: `d = {x: i for i, x in enumerate(arr)}` ánh xạ **giá trị -> chỉ số** (duy nhất). Tra `t in d` và `d[t]` đều `O(1)` trung bình.

In [3]:
arr = [1, 3, 7, 11, 12, 14, 18]
d = {x: i for i, x in enumerate(arr)}
print(d)
print("11 - 7 =", 11 - 7, "có trong arr?", (11 - 7) in d)
print("18 - 11 =", 18 - 11, "-> chỉ số", d.get(18 - 11))

{1: 0, 3: 1, 7: 2, 11: 3, 12: 4, 14: 5, 18: 6}
11 - 7 = 4 có trong arr? False
18 - 11 = 7 -> chỉ số 2


### 3.3. Vì sao phần tử trước phải nằm **trước** về chỉ số

Nếu `(arr[j], arr[i])` là hai phần tử cuối, phần tử trước là `t = arr[i] - arr[j]`. Để thành dãy con ta cần `t` đứng **trước** `arr[j]` trong `arr`. Vì `arr` tăng ngặt nên điều đó tương đương `t < arr[j]`, tức chỉ số `k` của `t` thỏa `k < j`. Nếu `t > arr[j]` thì `t` nằm **sau** `arr[j]` và không thể đứng trước nó trong dãy con.

In [4]:
arr = [1, 2, 3, 4, 5, 6, 7, 8]
d = {x: i for i, x in enumerate(arr)}
for j, i in [(3, 5), (4, 6), (1, 7)]:
    t = arr[i] - arr[j]
    k = d.get(t)
    print(f"cặp cuối ({arr[j]}, {arr[i]}): t = {t}, chỉ số k = {k}, hợp lệ (k < j)? {k is not None and k < j}")

cặp cuối (4, 6): t = 2, chỉ số k = 1, hợp lệ (k < j)? True
cặp cuối (5, 7): t = 2, chỉ số k = 1, hợp lệ (k < j)? True
cặp cuối (2, 8): t = 6, chỉ số k = 5, hợp lệ (k < j)? False


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với `arr = [1,3,7,11,12,14,18]`:

- Chọn hai phần tử đầu `(1, 3)`: phần tử kế `4` không có. Dãy dừng ở độ dài 2.
- Chọn `(3, 11)`: kế `14` có, rồi `25` không có: dãy `[3, 11, 14]` độ dài 3.
- Chọn `(1, 11)`: kế `12` có, rồi `23` không: `[1, 11, 12]` độ dài 3.
- Chọn `(7, 11)`: kế `18` có, rồi `29` không: `[7, 11, 18]` độ dài 3.

Có `n(n-1)/2` cặp đầu, mỗi cặp sinh một dãy duy nhất, ta chỉ cần xem dãy nào **dài nhất** (và có độ dài `>= 3`). Dãy cũng không thể dài quá `~44` phần tử vì giá trị tối đa `10^9` và mỗi số tối thiểu bằng số Fibonacci (nên mỗi vòng kéo dài rất ngắn).

In [5]:
arr = [1, 3, 7, 11, 12, 14, 18]
S = set(arr)
for a_idx in range(len(arr)):
    for b_idx in range(a_idx + 1, len(arr)):
        x, y = arr[a_idx], arr[b_idx]
        seq = [x, y]
        while seq[-1] + seq[-2] in S:
            seq.append(seq[-1] + seq[-2])
        if len(seq) >= 3:
            print(f"cặp đầu ({x}, {y}) -> {seq}")

cặp đầu (1, 11) -> [1, 11, 12]
cặp đầu (3, 11) -> [3, 11, 14]
cặp đầu (7, 11) -> [7, 11, 18]


## 5. Cách 1 — Brute Force: liệt kê mọi dãy con

**Ý tưởng.** Sinh mọi dãy con có độ dài `L >= 3` (tổ hợp chỉ số `combinations(arr, L)`; vì `arr` đã có thứ tự nên mỗi tổ hợp là một dãy con giữ thứ tự) và kiểm tra điều kiện Fibonacci. Đây là định nghĩa trực tiếp, nên cũng là **oracle**.

In [6]:
def is_fib_like(seq) -> bool:
    return len(seq) >= 3 and all(seq[t] + seq[t + 1] == seq[t + 2] for t in range(len(seq) - 2))

def longest_brute(arr: List[int]) -> int:
    best = 0
    n = len(arr)
    for L in range(3, n + 1):
        for comb in combinations(arr, L):
            if is_fib_like(comb):
                best = max(best, L)
    return best

for arr, expected in examples:
    got = longest_brute(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6, 7, 8] -> 5 OK
[1, 3, 7, 11, 12, 14, 18] -> 3 OK


**Độ phức tạp và nút thắt.** Có `2^n` dãy con: `n = 1000` là không thể. Hãy xem thời gian tăng theo `n`:

In [7]:
rng = random.Random(873)
for n in (10, 14, 16, 18, 20):
    arr = sorted(rng.sample(range(1, 200), n))
    t0 = time.perf_counter()
    longest_brute(arr)
    print(f"n={n}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n=10:       0.8 ms
n=14:       6.4 ms
n=16:      23.5 ms


n=18:     130.7 ms


n=20:     721.0 ms


Thời gian tăng gần **gấp đôi khi thêm một phần tử**. Nút thắt: ta xét cả những dãy con **chẳng bao giờ** có thể là Fibonacci. Ý tưởng cứu cánh: một dãy Fibonacci được xác định bởi **hai phần tử**, nên chỉ có `O(n^2)` khả năng.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát then chốt.** Một dãy kiểu Fibonacci bị **xác định hoàn toàn** bởi hai phần tử liên tiếp bất kỳ của nó. Do đó:

- **Hướng A (chọn cặp đầu):** với mỗi cặp `(i, j)`, `i < j`, tự sinh dãy `arr[i], arr[j], arr[i] + arr[j], ...` và đếm bao nhiêu phần tử đầu tiên nằm trong `arr` (dùng `set`). Các phần tử sinh ra **tăng dần** và `arr` tăng ngặt nên phần tử tìm thấy tự động nằm **sau** phần tử trước trong `arr`.
- **Hướng B (quy hoạch động theo cặp cuối):** gọi `f[i][j]` là độ dài dãy kiểu Fibonacci dài nhất kết thúc bằng `arr[j], arr[i]` (`j < i`). Phần tử trước là `t = arr[i] - arr[j]`; nếu `t` có trong `arr` ở chỉ số `k < j` thì `f[i][j] = f[j][k] + 1`, ngược lại `f[i][j] = 2`.

| Cách | Ý tưởng |
|---|---|
| 2 | Hướng A: mỗi cặp đầu kéo dài bằng `set` |
| 3 | Hướng B: DP hai chiều + bảng chỉ số (nộp) |
| 4 | Hướng B, nhưng tìm `k` bằng **hai con trỏ** thay vì bảng băm |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chọn cặp đầu rồi kéo dài bằng `set`

**Giải thích từng đoạn.**

- `S = set(arr)`: tra `O(1)`.
- Với mỗi cặp `i < j`: `a, b = arr[i], arr[j]`, `length = 2`. Lặp `while a + b in S`: `a, b = b, a + b`, `length += 1`.
- Chỉ khi `length >= 3` mới cập nhật đáp án (độ dài 2 không phải dãy kiểu Fibonacci).
- Vì `a + b > b` và `arr` tăng ngặt, phần tử `a + b` (nếu có) nằm sau `b` trong `arr` nên không cần kiểm tra thứ tự.
- **Cắt tỉa (không bắt buộc):** nếu `arr[i] + arr[j] > arr[-1]` thì dừng (cặp sau càng lớn hơn).

In [8]:
def longest_pair_extend(arr: List[int]) -> int:
    S = set(arr)
    n = len(arr)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            a, b = arr[i], arr[j]
            if a + b > arr[-1]:
                break
            length = 2
            while a + b in S:
                a, b = b, a + b
                length += 1
            if length >= 3:
                best = max(best, length)
    return best

for arr, expected in examples:
    got = longest_pair_extend(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6, 7, 8] -> 5 OK
[1, 3, 7, 11, 12, 14, 18] -> 3 OK


**Code nộp LeetCode (Cách 2):**

In [9]:
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        s = set(arr)
        n = len(arr)
        ans = 0
        for i in range(n):
            for j in range(i + 1, n):
                a, b = arr[i], arr[j]
                if a + b > arr[-1]:
                    break
                length = 2
                while a + b in s:
                    a, b = b, a + b
                    length += 1
                if length >= 3 and length > ans:
                    ans = length
        return ans

sol_extend = Solution().lenLongestFibSubseq
print([sol_extend(arr) for arr, _ in examples])
assert [sol_extend(arr) for arr, _ in examples] == [e for _, e in examples]

[5, 3]


**Độ phức tạp (Cách 2):** `O(n^2 * L)` với `L` là độ dài kéo dài tối đa (`<= ~44` vì giá trị `<= 10^9` và mỗi bước ít nhất nhân gần `1,6` lần); bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Quy hoạch động hai chiều + bảng chỉ số (cách nộp chính)

**Ý tưởng khác.** `f[i][j]` (`j < i`) là độ dài dãy kiểu Fibonacci dài nhất **kết thúc** bằng `arr[j], arr[i]`. Khởi tạo `f[i][j] = 2` (hai phần tử bất kỳ). Với `t = arr[i] - arr[j]`:

- nếu `t` có trong mảng ở chỉ số `k` và `k < j`: `f[i][j] = f[j][k] + 1` (nối thêm `arr[i]` vào dãy kết thúc bằng `arr[k], arr[j]`), và cập nhật đáp án;
- ngược lại giữ `2` (không đóng góp vào đáp án).

**Giải thích từng đoạn.**

- Thứ tự duyệt `i` tăng, `j < i`: `f[j][k]` (với `j < i`) luôn đã được tính vì `j < i` (hàng nhỏ hơn tính trước).
- `ans` chỉ cập nhật khi tìm được `k < j`, nên đáp án `>= 3` hoặc `0` (nếu không có chuỗi nào).

In [10]:
def longest_dp(arr: List[int]) -> int:
    n = len(arr)
    d = {x: i for i, x in enumerate(arr)}
    f = [[2] * n for _ in range(n)]
    ans = 0
    for i in range(2, n):
        for j in range(1, i):
            t = arr[i] - arr[j]
            k = d.get(t)
            if k is not None and k < j:
                f[i][j] = f[j][k] + 1
                ans = max(ans, f[i][j])
    return ans

for arr, expected in examples:
    got = longest_dp(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6, 7, 8] -> 5 OK
[1, 3, 7, 11, 12, 14, 18] -> 3 OK


**Code nộp LeetCode (Cách 3):**

In [11]:
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        n = len(arr)
        d = {x: i for i, x in enumerate(arr)}
        f = [[2] * n for _ in range(n)]
        ans = 0
        for i in range(2, n):
            for j in range(1, i):
                k = d.get(arr[i] - arr[j])
                if k is not None and k < j:
                    f[i][j] = f[j][k] + 1
                    if f[i][j] > ans:
                        ans = f[i][j]
        return ans

sol_dp = Solution().lenLongestFibSubseq
print([sol_dp(arr) for arr, _ in examples])
assert [sol_dp(arr) for arr, _ in examples] == [e for _, e in examples]

[5, 3]


**Độ phức tạp (Cách 3):** `O(n^2)` thời gian (mỗi cặp một lần tra bảng băm), `O(n^2)` bộ nhớ cho bảng `f`.

---

### 7.3. Cách 4 — Quy hoạch động với hai con trỏ (không cần bảng băm)

**Ý tưởng khác.** Cố định `i` làm **phần tử lớn nhất** (cuối) của dãy. Tìm mọi cặp `(k, j)` với `k < j < i` và `arr[k] + arr[j] = arr[i]` bằng hai con trỏ `lo = 0`, `hi = i - 1` trên mảng đã sắp xếp: nếu `arr[lo] + arr[hi] > arr[i]` thì `hi -= 1`; nếu `<` thì `lo += 1`; nếu `==` thì ta có cặp: cập nhật `f[i][hi] = f[hi][lo] + 1` rồi `lo += 1`, `hi -= 1`. Mỗi `i` tốn `O(n)`, tổng `O(n^2)` mà **không dùng bảng băm**.

**Giải thích từng đoạn.**

- `f[i][j] = 2` khi không tìm được `k` (khởi tạo mặc định bằng 2).
- Khi `arr[lo] + arr[hi] == arr[i]`: dãy kết thúc bằng `arr[lo], arr[hi], arr[i]`. Ta có `f[i][hi] = f[hi][lo] + 1`. Vì `hi < i`, `f[hi][lo]` đã được tính (hàng `hi` nhỏ hơn).
- Vì mảng tăng ngặt, mỗi `arr[hi]` ứng với **tối đa một** `lo` nên dịch cả hai con trỏ là đúng.

In [12]:
def longest_two_pointers(arr: List[int]) -> int:
    n = len(arr)
    f = [[2] * n for _ in range(n)]
    ans = 0
    for i in range(2, n):
        lo, hi = 0, i - 1
        while lo < hi:
            s = arr[lo] + arr[hi]
            if s > arr[i]:
                hi -= 1
            elif s < arr[i]:
                lo += 1
            else:
                f[i][hi] = f[hi][lo] + 1
                ans = max(ans, f[i][hi])
                lo += 1
                hi -= 1
    return ans

for arr, expected in examples:
    got = longest_two_pointers(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6, 7, 8] -> 5 OK
[1, 3, 7, 11, 12, 14, 18] -> 3 OK


**Code nộp LeetCode (Cách 4):**

In [13]:
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        n = len(arr)
        f = [[2] * n for _ in range(n)]
        ans = 0
        for i in range(2, n):
            lo, hi = 0, i - 1
            while lo < hi:
                s = arr[lo] + arr[hi]
                if s > arr[i]:
                    hi -= 1
                elif s < arr[i]:
                    lo += 1
                else:
                    f[i][hi] = f[hi][lo] + 1
                    if f[i][hi] > ans:
                        ans = f[i][hi]
                    lo += 1
                    hi -= 1
        return ans

sol_two = Solution().lenLongestFibSubseq
print([sol_two(arr) for arr, _ in examples])
assert [sol_two(arr) for arr, _ in examples] == [e for _, e in examples]

[5, 3]


**Độ phức tạp (Cách 4):** `O(n^2)` thời gian, `O(n^2)` bộ nhớ (bảng `f`).

## 8. TRACE từng bước

### 8.1. Cách 3 (DP) cho ví dụ 1: `arr = [1,2,3,4,5,6,7,8]`

Với mỗi cặp `(j, i)` mà `t = arr[i] - arr[j]` có chỉ số `k < j`, ta in `f[i][j] = f[j][k] + 1`. Chú ý chuỗi `1,2,3,5,8` hình thành khi `f` của các cặp cuối `(2,3) -> (3,5) -> (5,8)` tăng dần.

In [14]:
def trace_dp(arr):
    n = len(arr)
    d = {x: i for i, x in enumerate(arr)}
    f = [[2] * n for _ in range(n)]
    ans = 0
    for i in range(2, n):
        for j in range(1, i):
            t = arr[i] - arr[j]
            k = d.get(t)
            if k is not None and k < j:
                f[i][j] = f[j][k] + 1
                ans = max(ans, f[i][j])
                print(f"cặp cuối ({arr[j]}, {arr[i]}): t={t} ở chỉ số {k} < {j}  ->  f = f[{j}][{k}] + 1 = {f[i][j]}   (ans={ans})")
    return ans

assert trace_dp([1, 2, 3, 4, 5, 6, 7, 8]) == 5

cặp cuối (2, 3): t=1 ở chỉ số 0 < 1  ->  f = f[1][0] + 1 = 3   (ans=3)
cặp cuối (3, 4): t=1 ở chỉ số 0 < 2  ->  f = f[2][0] + 1 = 3   (ans=3)
cặp cuối (3, 5): t=2 ở chỉ số 1 < 2  ->  f = f[2][1] + 1 = 4   (ans=4)
cặp cuối (4, 5): t=1 ở chỉ số 0 < 3  ->  f = f[3][0] + 1 = 3   (ans=4)
cặp cuối (4, 6): t=2 ở chỉ số 1 < 3  ->  f = f[3][1] + 1 = 3   (ans=4)
cặp cuối (5, 6): t=1 ở chỉ số 0 < 4  ->  f = f[4][0] + 1 = 3   (ans=4)
cặp cuối (4, 7): t=3 ở chỉ số 2 < 3  ->  f = f[3][2] + 1 = 4   (ans=4)
cặp cuối (5, 7): t=2 ở chỉ số 1 < 4  ->  f = f[4][1] + 1 = 3   (ans=4)
cặp cuối (6, 7): t=1 ở chỉ số 0 < 5  ->  f = f[5][0] + 1 = 3   (ans=4)
cặp cuối (5, 8): t=3 ở chỉ số 2 < 4  ->  f = f[4][2] + 1 = 5   (ans=5)
cặp cuối (6, 8): t=2 ở chỉ số 1 < 5  ->  f = f[5][1] + 1 = 3   (ans=5)
cặp cuối (7, 8): t=1 ở chỉ số 0 < 6  ->  f = f[6][0] + 1 = 3   (ans=5)


Đường tăng của dãy `1, 2, 3, 5, 8`: cặp `(1,2)` có `f = 2`; `(2,3)` có `t = 1` ở chỉ số `0 < 1` nên `f = f[(1,2)] + 1 = 3`; `(3,5)` có `t = 2` ở chỉ số `1 < 2` nên `f = f[(2,3)] + 1 = 4`... và `(5,8)` có `t = 3` ở chỉ số `2 < 4` nên `f = f[(3,5)] + 1 = 5`.

### 8.2. Cách 2 (chọn cặp đầu) cho ví dụ 2

`arr = [1,3,7,11,12,14,18]`, `S` là tập các phần tử. Chỉ in những cặp đầu thật sự kéo dài được (độ dài `>= 3`).

In [15]:
def trace_extend(arr):
    S = set(arr)
    n = len(arr)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            a, b = arr[i], arr[j]
            seq = [a, b]
            while a + b in S:
                a, b = b, a + b
                seq.append(b)
            if len(seq) >= 3:
                best = max(best, len(seq))
                print(f"cặp đầu ({arr[i]}, {arr[j]}) -> {seq} (độ dài {len(seq)})")
    return best

assert trace_extend([1, 3, 7, 11, 12, 14, 18]) == 3

cặp đầu (1, 11) -> [1, 11, 12] (độ dài 3)
cặp đầu (3, 11) -> [3, 11, 14] (độ dài 3)
cặp đầu (7, 11) -> [7, 11, 18] (độ dài 3)


### 8.3. Cách 4 (hai con trỏ) cho `i` ứng với `arr[i] = 8` trong ví dụ 1

`arr = [1,2,3,4,5,6,7,8]`, `i = 7` (giá trị `8`), `lo = 0`, `hi = 6`:

- `1 + 7 = 8`: cặp `(1, 7)` -> `f[7][6] = f[6][0] + 1 = 3` (dãy `[1, 7, 8]`), rồi `lo = 1`, `hi = 5`.
- `2 + 6 = 8`: `f[7][5] = f[5][1] + 1 = 3` (dãy `[2, 6, 8]`; cặp `(2, 6)` không có phần tử trước nên `f[5][1] = 2`), rồi `lo = 2`, `hi = 4`.
- `3 + 5 = 8`: `f[7][4] = f[4][2] + 1`. Cặp `(3, 5)` đã được tính từ trước (lúc `i = 4`) là `f[4][2] = 4` (dãy `[1, 2, 3, 5]`), nên `f[7][4] = 5`, dãy `[1, 2, 3, 5, 8]`.
- `lo = 3`, `hi = 3`: dừng.

In [16]:
def trace_two_pointers(arr, target_index):
    n = len(arr)
    f = [[2] * n for _ in range(n)]
    for i in range(2, target_index + 1):
        lo, hi = 0, i - 1
        steps = []
        while lo < hi:
            s = arr[lo] + arr[hi]
            if s > arr[i]:
                steps.append(f"{arr[lo]}+{arr[hi]}={s}>{arr[i]}: hi-=1")
                hi -= 1
            elif s < arr[i]:
                steps.append(f"{arr[lo]}+{arr[hi]}={s}<{arr[i]}: lo+=1")
                lo += 1
            else:
                f[i][hi] = f[hi][lo] + 1
                steps.append(f"{arr[lo]}+{arr[hi]}={s}=={arr[i]}: f[{i}][{hi}] = f[{hi}][{lo}]+1 = {f[i][hi]}")
                lo += 1
                hi -= 1
        if i == target_index:
            print(f"i={i} (giá trị {arr[i]}):")
            for st in steps:
                print("   ", st)
    return f

f = trace_two_pointers([1, 2, 3, 4, 5, 6, 7, 8], 7)
print("dãy kết thúc (5, 8) có độ dài", f[7][4], "-> [1,2,3,5,8]")
assert f[7][4] == 5

i=7 (giá trị 8):
    1+7=8==8: f[7][6] = f[6][0]+1 = 3
    2+6=8==8: f[7][5] = f[5][1]+1 = 3
    3+5=8==8: f[7][4] = f[4][2]+1 = 5
dãy kết thúc (5, 8) có độ dài 5 -> [1,2,3,5,8]


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (quy hoạch động hai chiều với bảng chỉ số): ý tưởng "trạng thái là hai phần tử cuối" rõ ràng, `O(n^2)` thời gian và bộ nhớ; Cách 2 thường nhanh hơn thực tế nhờ cắt tỉa.

```python
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        n = len(arr)
        d = {x: i for i, x in enumerate(arr)}
        f = [[2] * n for _ in range(n)]
        ans = 0
        for i in range(2, n):
            for j in range(1, i):
                k = d.get(arr[i] - arr[j])
                if k is not None and k < j:
                    f[i][j] = f[j][k] + 1
                    if f[i][j] > ans:
                        ans = f[i][j]
        return ans
```

In [17]:
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        n = len(arr)
        d = {x: i for i, x in enumerate(arr)}
        f = [[2] * n for _ in range(n)]
        ans = 0
        for i in range(2, n):
            for j in range(1, i):
                k = d.get(arr[i] - arr[j])
                if k is not None and k < j:
                    f[i][j] = f[j][k] + 1
                    if f[i][j] > ans:
                        ans = f[i][j]
        return ans

sol = Solution()
for arr, expected in examples:
    got = sol.lenLongestFibSubseq(arr)
    print(arr, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1, 2, 3, 4, 5, 6, 7, 8] -> 5 OK
[1, 3, 7, 11, 12, 14, 18] -> 3 OK


## 10. Vì sao đúng

**(a) Dãy kiểu Fibonacci được xác định bởi hai phần tử liên tiếp.** Cho `(x, y)`: phần tử kế là `x + y`, rồi `y + (x + y)`, ... Cho `(y, z)`: phần tử trước là `z - y`. Vậy mỗi dãy kiểu Fibonacci tương ứng **duy nhất** với cặp hai phần tử đầu của nó.

**(b) Cách 2.** Với cặp đầu `(a, b)` ta sinh dãy duy nhất `a, b, a + b, ...`. Dãy là dãy con của `arr` khi và chỉ khi mỗi phần tử sinh ra có trong `arr` **và** thứ tự chỉ số tăng. Các phần tử sinh ra **tăng ngặt** (`a < b`, `a + b > b`, ...) và `arr` tăng ngặt, nên phần tử có giá trị lớn hơn luôn nằm ở chỉ số lớn hơn; do đó điều kiện thứ tự tự động đúng nếu mỗi phần tử có trong `arr`. Dãy dài nhất bắt đầu từ cặp đó là số phần tử liên tiếp có mặt trong `arr`; lấy `max` theo mọi cặp (và chỉ tính độ dài `>= 3`) cho đáp án.

**(c) Cách 3 (quy hoạch động).** Định nghĩa `f[i][j]` là độ dài dãy kiểu Fibonacci dài nhất (cho phép độ dài 2) kết thúc bằng `arr[j], arr[i]`, `j < i`.
   - Cơ sở: `f[i][j] >= 2` (hai phần tử bất kỳ).
   - Chuyển trạng thái: nếu dãy có độ dài `>= 3` kết thúc bằng `arr[j], arr[i]` thì phần tử trước `arr[j]` bắt buộc là `t = arr[i] - arr[j]` (tính duy nhất). `t` có trong `arr` (giá trị duy nhất vì `arr` tăng ngặt) ở chỉ số `k`; để thành dãy con cần `k < j` (tương đương `t < arr[j]`). Phần còn lại của dãy là một dãy kết thúc bằng `arr[k], arr[j]` với độ dài tối đa `f[j][k]`; vậy `f[i][j] = f[j][k] + 1`.
   - Nếu không có `t` hợp lệ thì chỉ có dãy độ dài 2.
   - Phụ thuộc: `f[i][j]` cần `f[j][k]` với `j < i`: hàng nhỏ hơn đã tính. Đáp án là `max` các `f[i][j]` có `t` hợp lệ (độ dài `>= 3`); nếu không có thì `0`.

**(d) Cách 4.** Điều kiện `arr[k] + arr[j] = arr[i]` với `k < j < i`: hai con trỏ trên mảng tăng ngặt liệt kê mọi cặp như vậy (bài toán hai tổng 2-sum): nếu tổng lớn hơn thì bỏ `arr[hi]` (mọi `lo` lớn hơn chỉ làm tổng lớn thêm), nếu nhỏ hơn thì bỏ `arr[lo]`. Mỗi cặp khớp cho cùng chuyển trạng thái như Cách 3.

**Kiểm chứng bằng máy.** Hai oracle **độc lập** với các lời giải trên: (1) liệt kê mọi tập con (`combinations`) và kiểm tra định nghĩa (`n <= 14`); (2) DFS theo **vị trí**: với mỗi cặp chỉ số đầu `(i, j)` quét tuyến tính phần còn lại của mảng để tìm giá trị `a + b` (không dùng `set`, không dùng bảng băm) (`n <= 60`). So sánh mọi cách trên ví dụ, edge case, mảng ngẫu nhiên và mảng được xây từ nhiều dãy Fibonacci.

In [18]:
def oracle_subsets(arr: List[int]) -> int:
    return longest_brute(arr)

def oracle_scan(arr: List[int]) -> int:
    """Với mỗi cặp chỉ số đầu (i, j), tìm phần tử kế bằng quét tuyến tính theo vị trí."""
    n = len(arr)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):
            a, b, pos, length = arr[i], arr[j], j + 1, 2
            while True:
                nxt = a + b
                while pos < n and arr[pos] < nxt:
                    pos += 1
                if pos < n and arr[pos] == nxt:
                    a, b, length, pos = b, nxt, length + 1, pos + 1
                else:
                    break
            if length >= 3:
                best = max(best, length)
    return best

def random_arr(rng, n, vmax):
    return sorted(rng.sample(range(1, vmax + 1), n))

rng = random.Random(1873)
for _ in range(200):
    n = rng.randint(3, 12)
    arr = random_arr(rng, n, rng.choice([n + 2, 20, 60]))
    assert oracle_subsets(arr) == oracle_scan(arr), arr
print("hai oracle (tập con / quét vị trí) khớp nhau trên 200 mảng ngẫu nhiên")

hai oracle (tập con / quét vị trí) khớp nhau trên 200 mảng ngẫu nhiên


## 11. Time / Space Complexity

Gọi `n` là độ dài `arr`, `L` là độ dài kéo dài tối đa (`<= ~44`).

| Cách | Time | Space |
|---|---|---|
| 1. Mọi dãy con | `O(2^n * n)` | `O(n)` |
| 2. Cặp đầu + `set` | `O(n^2 * L)` | `O(n)` |
| 3. DP 2D + bảng chỉ số | `O(n^2)` | `O(n^2)` |
| 4. DP 2D + hai con trỏ | `O(n^2)` | `O(n^2)` |

Đo thử với `n = 1000` (ba dạng dữ liệu: ngẫu nhiên thưa, dày `1..1000`, nhiều dãy Fibonacci):

In [19]:
rng = random.Random(11)
def many_fibs(rng, count, vmax=10 ** 9):
    values = set()
    for _ in range(count):
        a, b = rng.randint(1, 50), rng.randint(1, 50)
        while b <= vmax:
            values.add(b)
            a, b = b, a + b
    return sorted(values)

big_random = random_arr(rng, 1000, 10 ** 9)
dense = list(range(1, 1001))
fibs = many_fibs(rng, 400)
if len(fibs) > 1000:
    fibs = sorted(rng.sample(fibs, 1000))
datasets = {"ngẫu nhiên thưa": big_random, "dày 1..1000": dense, f"nhiều dãy Fibonacci ({len(fibs)} phần tử)": fibs}
for name, arr in datasets.items():
    results = {}
    for label, fn in (("Cách 2", sol_extend), ("Cách 3", sol_dp), ("Cách 4", sol_two)):
        t0 = time.perf_counter()
        results[label] = fn(arr)
        results[label + " ms"] = round(1000 * (time.perf_counter() - t0))
    assert results["Cách 2"] == results["Cách 3"] == results["Cách 4"], name
    print(f"{name:42} kết quả {results['Cách 2']:3} | " + " | ".join(f"{k}: {results[k + ' ms']} ms" for k in ("Cách 2", "Cách 3", "Cách 4")))

ngẫu nhiên thưa                            kết quả   0 | Cách 2: 32 ms | Cách 3: 45 ms | Cách 4: 69 ms


dày 1..1000                                kết quả  15 | Cách 2: 82 ms | Cách 3: 81 ms | Cách 4: 64 ms


nhiều dãy Fibonacci (1000 phần tử)         kết quả   5 | Cách 2: 95 ms | Cách 3: 61 ms | Cách 4: 72 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với hai oracle, và test ngẫu nhiên có seed (đúng ràng buộc `3 <= n <= 1000`, giá trị tăng ngặt trong `[1, 10^9]`).

In [20]:
def solutions_list():
    return {
        "pair_extend": longest_pair_extend,
        "dp": longest_dp,
        "two_pointers": longest_two_pointers,
        "sol_extend": sol_extend,
        "sol_dp": sol_dp,
        "sol_two": sol_two,
        "final": Solution().lenLongestFibSubseq,
    }

def run_all(arr, expected=None, oracle=None):
    results = {name: fn(list(arr)) for name, fn in solutions_list().items()}
    ref = oracle if oracle is not None else oracle_scan(arr)
    for name, got in results.items():
        assert got == ref, (name, arr, got, ref)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", arr, expected, ref)
    return ref

# 12.1 ví dụ đề
for arr, expected in examples:
    assert run_all(arr, expected) == expected
    assert longest_brute(arr) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([1, 2, 3], 3),                         # đúng một dãy 1+2=3
    ([1, 2, 4], 0),                         # 1+2=3 không có, không dãy nào
    ([1, 3, 4], 3),                         # 1+3=4
    ([2, 3, 5], 3),
    ([1, 2, 3, 5, 8, 13, 21], 7),           # cả mảng là Fibonacci
    ([1, 2, 3, 5, 8, 13, 21, 34, 55, 89], 10),
    ([1, 3, 5, 7, 9], 0),                   # toàn số lẻ: tổng hai lẻ là chẵn
    ([5, 6, 7], 0),                         # 5+6 = 11 không có
    ([1, 2, 3, 4], 3),                      # 1,2,3 hoặc 1,3,4
    ([1, 10 ** 9 - 1, 10 ** 9], 3),         # biên lớn: 1 + (1e9-1) = 1e9
    ([10 ** 9 - 2, 10 ** 9 - 1, 10 ** 9], 0),
    ([1, 2, 3, 4, 5, 6, 7, 8], 5),
    ([3, 4, 7, 11, 18, 29], 6),             # Lucas: 3,4,7,11,18,29
    ([2, 4, 7, 10, 11, 14, 18], 4),         # 4, 7, 11, 18
]
for arr, expected in edge:
    assert run_all(arr, expected) == expected, arr
    if len(arr) <= 14:
        assert longest_brute(arr) == expected, ("vét cạn khác kỳ vọng", arr)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 14 ca


In [21]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_min_two(arr):                       # trả về 2 thay vì 0 khi không có dãy độ dài >= 3
    n = len(arr)
    d = {x: i for i, x in enumerate(arr)}
    f = [[2] * n for _ in range(n)]
    ans = 2
    for i in range(2, n):
        for j in range(1, i):
            k = d.get(arr[i] - arr[j])
            if k is not None and k < j:
                f[i][j] = f[j][k] + 1
                ans = max(ans, f[i][j])
    return ans

def wrong_no_order_check(arr):                # quên điều kiện k < j
    n = len(arr)
    d = {x: i for i, x in enumerate(arr)}
    f = [[2] * n for _ in range(n)]
    ans = 0
    for i in range(2, n):
        for j in range(1, i):
            k = d.get(arr[i] - arr[j])
            if k is not None and k != j:
                f[i][j] = f[j][k] + 1
                ans = max(ans, f[i][j])
    return ans

def wrong_first_pair_only(arr):               # chỉ thử cặp đầu tiên của mảng làm hai phần tử đầu
    S = set(arr)
    a, b = arr[0], arr[1]
    length = 2
    while a + b in S:
        a, b = b, a + b
        length += 1
    return length if length >= 3 else 0

def wrong_consecutive_pairs(arr):             # chỉ thử các cặp đầu liền kề (i, i+1)
    S = set(arr)
    best = 0
    for i in range(len(arr) - 1):
        a, b = arr[i], arr[i + 1]
        length = 2
        while a + b in S:
            a, b = b, a + b
            length += 1
        if length >= 3:
            best = max(best, length)
    return best

def find_counter(wrong, tries=20000, seed=0):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(3, 8)
        arr = random_arr(rng, n, rng.choice([8, 15, 30]))
        if wrong(arr) != oracle_scan(arr):
            return arr
    return None

counters = {}
for name, fn in (("trả 2 thay vì 0", wrong_min_two), ("quên điều kiện k < j", wrong_no_order_check),
                 ("chỉ thử cặp đầu của mảng", wrong_first_pair_only), ("chỉ thử cặp đầu liền kề", wrong_consecutive_pairs)):
    arr = find_counter(fn)
    assert arr is not None, name
    counters[name] = arr
    print(f"{name:28} arr={arr}: sai = {fn(arr)}, đúng = {oracle_scan(arr)}")
    assert run_all(arr) == oracle_scan(arr)           # các lời giải đúng không bị lỗi
print("12.3 phản ví dụ: OK")

trả 2 thay vì 0              arr=[9, 26, 30]: sai = 2, đúng = 0
quên điều kiện k < j         arr=[1, 2, 4, 6, 7, 8]: sai = 4, đúng = 3
chỉ thử cặp đầu của mảng     arr=[1, 5, 7, 8, 9, 14]: sai = 0, đúng = 3
chỉ thử cặp đầu liền kề      arr=[1, 5, 7, 8, 9, 14]: sai = 0, đúng = 3
12.3 phản ví dụ: OK


In [22]:
# 12.4 ngẫu nhiên nhỏ: so mọi cách với oracle tập con (n <= 12) và oracle quét vị trí
rng = random.Random(4873)
for _ in range(600):
    n = rng.randint(3, 12)
    arr = random_arr(rng, n, rng.choice([n + 1, 15, 40, 200]))
    assert run_all(arr, None, oracle_subsets(arr)) == oracle_scan(arr)
print("12.4 ngẫu nhiên n <= 12 (so với liệt kê tập con): OK")

# 12.5 trung bình: n tới 60, giá trị nhỏ để có nhiều dãy Fibonacci (so với oracle quét vị trí)
rng = random.Random(5873)
for _ in range(300):
    n = rng.randint(13, 60)
    arr = random_arr(rng, n, rng.choice([n + 5, 100, 300, 10 ** 9]))
    run_all(arr)
print("12.5 ngẫu nhiên n tới 60: OK")

# 12.6 mảng xây từ nhiều dãy Fibonacci (chắc chắn có dãy dài), đúng ràng buộc
rng = random.Random(6873)
for _ in range(60):
    arr = many_fibs(rng, rng.randint(1, 6), vmax=rng.choice([100, 10 ** 4, 10 ** 9]))
    if len(arr) < 3:
        continue
    run_all(arr)
    if len(arr) > 60:
        got = Solution().lenLongestFibSubseq(arr)
        assert got == longest_dp(arr) == longest_two_pointers(arr) == longest_pair_extend(arr)
print("12.6 mảng nhiều dãy Fibonacci: OK")

# 12.7 mảng lớn n = 1000: ba cách nhanh khớp nhau và khớp oracle quét cho vài mảng
rng = random.Random(7873)
for vmax in (1000, 2000, 10 ** 5, 10 ** 9):
    arr = random_arr(rng, 1000, vmax)
    r = {name: fn(arr) for name, fn in (("extend", sol_extend), ("dp", sol_dp), ("two", sol_two), ("final", Solution().lenLongestFibSubseq))}
    assert len(set(r.values())) == 1, (vmax, r)
    print(f"  n=1000, giá trị tới {vmax:>10}: đáp án {r['final']}")
print("12.7 n = 1000: OK")

# 12.8 tính chất: thêm một phần tử không làm giảm đáp án; bỏ phần tử cuối không làm tăng
rng = random.Random(8873)
for _ in range(300):
    n = rng.randint(4, 30)
    arr = random_arr(rng, n, 80)
    base = Solution().lenLongestFibSubseq(arr)
    assert Solution().lenLongestFibSubseq(arr[:-1]) <= base
print("12.8 tính chất đơn điệu theo tập con: OK")

12.4 ngẫu nhiên n <= 12 (so với liệt kê tập con): OK


12.5 ngẫu nhiên n tới 60: OK


12.6 mảng nhiều dãy Fibonacci: OK


  n=1000, giá trị tới       1000: đáp án 15


  n=1000, giá trị tới       2000: đáp án 9


  n=1000, giá trị tới     100000: đáp án 4
  n=1000, giá trị tới 1000000000: đáp án 0
12.7 n = 1000: OK
12.8 tính chất đơn điệu theo tập con: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về `2` thay vì `0` khi không có dãy độ dài `>= 3`

Đề yêu cầu trả `0` nếu không có dãy kiểu Fibonacci (độ dài tối thiểu `3`). Khởi tạo `ans = 2` (vì mọi cặp là "độ dài 2") là sai.

In [23]:
arr = counters["trả 2 thay vì 0"]
print("arr =", arr, "| sai ->", wrong_min_two(arr), "| đúng ->", Solution().lenLongestFibSubseq(arr))
assert wrong_min_two(arr) == 2 and Solution().lenLongestFibSubseq(arr) == 0
print("ví dụ cố định: [1, 2, 4] ->", wrong_min_two([1, 2, 4]), "(sai) so với", Solution().lenLongestFibSubseq([1, 2, 4]), "(đúng)")

arr = [9, 26, 30] | sai -> 2 | đúng -> 0
ví dụ cố định: [1, 2, 4] -> 2 (sai) so với 0 (đúng)


Vì sao sai: bảng DP mặc định `2` nhưng đáp án chỉ cập nhật khi tìm được `k` hợp lệ; nếu khởi tạo `ans = 2` thì mảng không có dãy nào cũng bị trả `2`. **Đúng:** `ans = 0` và cập nhật chỉ khi có dãy độ dài `>= 3`.

### Lỗi 2 — Quên điều kiện thứ tự `k < j`

`t = arr[i] - arr[j]` có thể có trong `arr` nhưng ở **sau** `arr[j]` (khi `t > arr[j]`), khi đó dãy sẽ không còn giữ thứ tự (phần tử "trước" lại đứng sau).

In [24]:
arr = counters["quên điều kiện k < j"]
print("arr =", arr, "| sai ->", wrong_no_order_check(arr), "| đúng ->", Solution().lenLongestFibSubseq(arr))
assert wrong_no_order_check(arr) != Solution().lenLongestFibSubseq(arr)

arr = [1, 2, 4, 6, 7, 8] | sai -> 4 | đúng -> 3


Vì sao sai: nếu `t > arr[j]` thì `t` nằm giữa `arr[j]` và `arr[i]`; dãy `t, arr[j], arr[i]` giảm rồi tăng và không phải dãy con (thứ tự bị đảo). **Đúng:** yêu cầu `k < j` (hay `t < arr[j]`).

### Lỗi 3 — Chỉ thử một cặp đầu hoặc chỉ các cặp liền kề

Dãy con không cần liền kề, và hai phần tử đầu của dãy tối ưu có thể là **bất kỳ** hai phần tử của `arr`.

In [25]:
for name in ("chỉ thử cặp đầu của mảng", "chỉ thử cặp đầu liền kề"):
    arr = counters[name]
    fn = wrong_first_pair_only if "của mảng" in name else wrong_consecutive_pairs
    print(f"{name}: arr = {arr} | sai -> {fn(arr)} | đúng -> {Solution().lenLongestFibSubseq(arr)}")
    assert fn(arr) != Solution().lenLongestFibSubseq(arr)

chỉ thử cặp đầu của mảng: arr = [1, 5, 7, 8, 9, 14] | sai -> 0 | đúng -> 3
chỉ thử cặp đầu liền kề: arr = [1, 5, 7, 8, 9, 14] | sai -> 0 | đúng -> 3


Vì sao sai: bỏ sót các dãy bắt đầu ở cặp không liền kề, ví dụ `[1, 11, 12]` trong ví dụ 2 (`1` và `11` không liền kề). **Đúng:** thử **mọi** cặp `(i, j)` (`O(n^2)`) hoặc dùng DP trên mọi cặp cuối.

### Lỗi 4 — Dùng `list.index` hoặc tìm tuyến tính thay cho bảng băm

Không sai kết quả nhưng biến `O(n^2)` thành `O(n^3)`: với `n = 1000` là hàng tỉ thao tác, **TLE**.

In [26]:
def dp_with_linear_search(arr):
    n = len(arr)
    f = [[2] * n for _ in range(n)]
    ans = 0
    for i in range(2, n):
        for j in range(1, i):
            t = arr[i] - arr[j]
            if t in arr:                      # tìm tuyến tính trong list: O(n)
                k = arr.index(t)
                if k < j:
                    f[i][j] = f[j][k] + 1
                    ans = max(ans, f[i][j])
    return ans

rng = random.Random(13)
arr = random_arr(rng, 400, 10 ** 9)
t0 = time.perf_counter(); a1 = dp_with_linear_search(arr); t1 = time.perf_counter() - t0
t0 = time.perf_counter(); a2 = Solution().lenLongestFibSubseq(arr); t2 = time.perf_counter() - t0
assert a1 == a2
print(f"n=400: tuyến tính {1000 * t1:.0f} ms | bảng băm {1000 * t2:.0f} ms (chênh lệch tăng theo n)")

n=400: tuyến tính 180 ms | bảng băm 5 ms (chênh lệch tăng theo n)


Vì sao chậm: `t in arr` và `arr.index` đều `O(n)`. **Đúng:** `d = {x: i for i, x in enumerate(arr)}` rồi `d.get(t)` `O(1)`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** dãy con có **quy luật hai phần tử xác định tất cả** (Fibonacci, cấp số cộng cố định) ⇒ trạng thái là **cặp hai phần tử cuối** `(j, i)` hoặc **cặp đầu**.
- **Kỹ thuật:** bảng băm `giá trị -> chỉ số`; kiểm tra thứ tự bằng `k < j` hoặc bằng giá trị (do mảng tăng ngặt).
- Độ dài dãy tối đa chỉ cỡ `~44` với giá trị `<= 10^9`, nên "cặp đầu + `while`" rất nhanh.
- Khởi tạo DP bằng `2` (hai phần tử bất kỳ) nhưng **chỉ ghi nhận đáp án** khi `>= 3`.
- Cùng họ: Longest Arithmetic Subsequence (1027), Longest Arithmetic Subsequence of Given Difference (1218), Number of Arithmetic Triplets, Arithmetic Slices II (446), Longest Increasing Subsequence (300).
- Test nên có: `[1,2,4]` (đáp án `0`), cả mảng là Fibonacci, Lucas `[3,4,7,11,18,29]`, biên `10^9`, mảng dày `1..1000`.

## 15. Biến thể

1. **Trả về chính dãy dài nhất** (không chỉ độ dài): truy vết từ cặp cuối tốt nhất.
2. **Đếm** số dãy con kiểu Fibonacci (bài tập 5).
3. **Mảng không tăng ngặt** hoặc có phần tử trùng: cần bảng `giá trị -> danh sách chỉ số` hoặc DP theo `(giá trị, giá trị)`.
4. Bài liên quan: Arithmetic Slices II - Subsequence (446), Longest Arithmetic Subsequence (1027).

Dưới đây là biến thể 1 (trả về dãy thật) kiểm bằng hàm kiểm tra hợp lệ.

In [27]:
def longest_sequence(arr: List[int]) -> List[int]:
    S = set(arr)
    n = len(arr)
    best_seq: List[int] = []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = arr[i], arr[j]
            seq = [a, b]
            while a + b in S:
                a, b = b, a + b
                seq.append(b)
            if len(seq) >= 3 and len(seq) > len(best_seq):
                best_seq = seq
    return best_seq

def valid_answer(arr, seq):
    pos = {x: i for i, x in enumerate(arr)}
    return (is_fib_like(seq) and all(x in pos for x in seq)
            and all(pos[seq[t]] < pos[seq[t + 1]] for t in range(len(seq) - 1)))

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(3, 25)
    arr = random_arr(rng, n, rng.choice([30, 100, 1000]))
    seq = longest_sequence(arr)
    L = oracle_scan(arr)
    assert len(seq) == L, (arr, seq, L)
    if L:
        assert valid_answer(arr, seq), (arr, seq)
print(longest_sequence([1, 2, 3, 4, 5, 6, 7, 8]), longest_sequence([1, 3, 7, 11, 12, 14, 18]), longest_sequence([1, 2, 4]))
print("Biến thể: OK")

[1, 2, 3, 5, 8] [1, 11, 12] []
Biến thể: OK


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi dãy con | Định nghĩa | `O(2^n n)` | `O(n)` | Oracle, `n <= 14` |
| 2. Cặp đầu + `set` | Dãy xác định bởi cặp đầu | `O(n^2 L)` | `O(n)` | Đơn giản, bộ nhớ nhỏ, thường nhanh |
| 3. DP 2D + bảng chỉ số | Trạng thái cặp cuối | `O(n^2)` | `O(n^2)` | **Cách nộp chính**, mở rộng được (đếm, truy vết) |
| 4. DP 2D + hai con trỏ | Tìm `k` bằng 2-sum | `O(n^2)` | `O(n^2)` | Không cần bảng băm |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
arr = [1, 2, 3, 5, 8]
d = {x: i for i, x in enumerate(arr)}
print(d[8 - 5], (8 - 3) in d, d.get(8 - 2))
print(extend(2, 3, 3))
print(is_fib_like([2, 3, 5, 8]), is_fib_like([2, 3, 6]))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `arr = [2, 4, 7, 10, 11, 14, 18]`: ghi độ dài dãy kiểu Fibonacci dài nhất và dãy đó.

In [29]:
# Bài 2: điền kết quả tính tay
hand_length = None
hand_sequence = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành cách "chọn cặp đầu rồi kéo dài".

In [30]:
def exercise_3_extend(arr: List[int]) -> int:
    s = set(arr)
    n = len(arr)
    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            a, b = arr[i], arr[j]
            length = 2
            # TODO 1: while a + b nằm trong s: cập nhật a, b = b, a + b và tăng length
            # TODO 2: nếu length >= 3 thì cập nhật ans
            pass
    return ans


# assert exercise_3_extend([1, 2, 3, 4, 5, 6, 7, 8]) == 5
# assert exercise_3_extend([1, 3, 7, 11, 12, 14, 18]) == 3
# assert exercise_3_extend([1, 2, 4]) == 0

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_dp(arr)` dùng DP với **từ điển khóa là cặp giá trị** `(arr[j], arr[i])` thay cho bảng 2 chiều: `dp[(a, b)]` là độ dài dãy dài nhất kết thúc bằng `a, b`.

In [31]:
def exercise_4_dp(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_dp([1, 2, 3, 4, 5, 6, 7, 8]) == 5
# assert exercise_4_dp([1, 3, 7, 11, 12, 14, 18]) == 3
# assert exercise_4_dp([1, 2, 4]) == 0

### Bài 5 — Mini challenge: đếm số dãy con kiểu Fibonacci (khó)

Viết `exercise_5_count(arr)` trả về **số dãy con** (khác nhau theo **tập chỉ số**) kiểu Fibonacci có độ dài `>= 3`.

```python
exercise_5_count([1, 2, 3])              -> 1     # [1,2,3]
exercise_5_count([1, 2, 3, 4, 5, 6, 7, 8]) -> 16
exercise_5_count([1, 2, 4])              -> 0
```
Gợi ý: gọi `g[i][j]` là số dãy (độ dài `>= 2`) kết thúc bằng `arr[j], arr[i]`; `g[i][j] = 1 + g[j][k]` nếu có `k < j`, và mỗi dãy độ dài `>= 3` kết thúc bằng `(j, i)` đếm bằng `g[j][k]`.

In [32]:
def exercise_5_count(arr: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_count([1, 2, 3]) == 1
# assert exercise_5_count([1, 2, 3, 4, 5, 6, 7, 8]) == 16
# assert exercise_5_count([1, 2, 4]) == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `d` ánh xạ giá trị -> chỉ số; `extend(x, y, count)` thêm `count` phần tử.
- Mức 2: `8 - 5 = 3` ở chỉ số `2`; `8 - 3 = 5` có; `8 - 2 = 6` không có.
- Mức 3: `2 True None`, `[2, 3, 5, 8, 13]`, `True False`.

**Bài 2**
- Mức 1: thử cặp `(4, 7)`: `4 + 7 = 11`.
- Mức 2: tiếp `7 + 11 = 18`; tiếp `11 + 18 = 29` không có.
- Mức 3: độ dài `4`, dãy `[4, 7, 11, 18]`.

**Bài 3**
- Mức 1: `while a + b in s:` kéo dài từng bước.
- Mức 2: `a, b = b, a + b; length += 1`.
- Mức 3: sau vòng `while`: `if length >= 3 and length > ans: ans = length`.

**Bài 4**
- Mức 1: `d = {x: i}`; với mỗi cặp `(j, i)`, `t = arr[i] - arr[j]`.
- Mức 2: nếu `t` có và `d[t] < j`: `dp[(arr[j], arr[i])] = dp.get((t, arr[j]), 2) + 1`.
- Mức 3: `ans = max(ans, dp[(arr[j], arr[i])])`; ban đầu `ans = 0`.

**Bài 5**
- Mức 1: số dãy độ dài `>= 3` kết thúc bằng `(j, i)` bằng số dãy độ dài `>= 2` kết thúc bằng `(k, j)`.
- Mức 2: `g[i][j] = 1 + g[j][k]` khi có `k`, ngược lại `1`; cộng `g[j][k]` vào đáp án.
- Mức 3: `total += g[j][k]`; đáp án là tổng.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
arr = [1, 2, 3, 5, 8]
d = {x: i for i, x in enumerate(arr)}
print(d[8 - 5], (8 - 3) in d, d.get(8 - 2))
print(extend(2, 3, 3))
print(is_fib_like([2, 3, 5, 8]), is_fib_like([2, 3, 6]))

assert (d[8 - 5], (8 - 3) in d, d.get(8 - 2)) == (2, True, None)
assert extend(2, 3, 3) == [2, 3, 5, 8, 13]
assert (is_fib_like([2, 3, 5, 8]), is_fib_like([2, 3, 6])) == (True, False)

2 True None
[2, 3, 5, 8, 13]
True False


### Đáp án Bài 2

`4 + 7 = 11`, `7 + 11 = 18`, `11 + 18 = 29` không có: dãy `[4, 7, 11, 18]` dài `4`.

In [34]:
hand_length_solution = 4
hand_sequence_solution = [4, 7, 11, 18]
arr = [2, 4, 7, 10, 11, 14, 18]
assert Solution().lenLongestFibSubseq(arr) == hand_length_solution == oracle_scan(arr) == oracle_subsets(arr)
assert valid_answer(arr, hand_sequence_solution) and len(hand_sequence_solution) == hand_length_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_extend_solution(arr: List[int]) -> int:
    s = set(arr)
    n = len(arr)
    ans = 0
    for i in range(n):
        for j in range(i + 1, n):
            a, b = arr[i], arr[j]
            length = 2
            while a + b in s:
                a, b = b, a + b
                length += 1
            if length >= 3 and length > ans:
                ans = length
    return ans

assert exercise_3_extend_solution([1, 2, 3, 4, 5, 6, 7, 8]) == 5
assert exercise_3_extend_solution([1, 3, 7, 11, 12, 14, 18]) == 3
assert exercise_3_extend_solution([1, 2, 4]) == 0
rng = random.Random(3)
for _ in range(300):
    arr = random_arr(rng, rng.randint(3, 25), rng.choice([30, 100, 1000]))
    assert exercise_3_extend_solution(arr) == oracle_scan(arr)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_dp_solution(arr: List[int]) -> int:
    d = {x: i for i, x in enumerate(arr)}
    dp = {}
    ans = 0
    for i in range(len(arr)):
        for j in range(i):
            t = arr[i] - arr[j]
            k = d.get(t)
            if k is not None and k < j:
                dp[(arr[j], arr[i])] = dp.get((t, arr[j]), 2) + 1
                ans = max(ans, dp[(arr[j], arr[i])])
    return ans

assert exercise_4_dp_solution([1, 2, 3, 4, 5, 6, 7, 8]) == 5
assert exercise_4_dp_solution([1, 3, 7, 11, 12, 14, 18]) == 3
assert exercise_4_dp_solution([1, 2, 4]) == 0
rng = random.Random(4)
for _ in range(300):
    arr = random_arr(rng, rng.randint(3, 25), rng.choice([30, 100, 1000]))
    assert exercise_4_dp_solution(arr) == oracle_scan(arr)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_count_solution(arr: List[int]) -> int:
    n = len(arr)
    d = {x: i for i, x in enumerate(arr)}
    g = [[1] * n for _ in range(n)]
    total = 0
    for i in range(2, n):
        for j in range(1, i):
            k = d.get(arr[i] - arr[j])
            if k is not None and k < j:
                g[i][j] = 1 + g[j][k]
                total += g[j][k]
    return total

def count_brute(arr: List[int]) -> int:
    return sum(1 for L in range(3, len(arr) + 1) for comb in combinations(arr, L) if is_fib_like(comb))

assert exercise_5_count_solution([1, 2, 3]) == 1
assert exercise_5_count_solution([1, 2, 3, 4, 5, 6, 7, 8]) == count_brute([1, 2, 3, 4, 5, 6, 7, 8]) == 16
assert exercise_5_count_solution([1, 2, 4]) == 0
rng = random.Random(5)
for _ in range(300):
    arr = random_arr(rng, rng.randint(3, 13), rng.choice([15, 40, 200]))
    assert exercise_5_count_solution(arr) == count_brute(arr), arr
print("Bài 5 OK; số dãy con của [1..8]:", exercise_5_count_solution([1, 2, 3, 4, 5, 6, 7, 8]))

Bài 5 OK; số dãy con của [1..8]: 16


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (DP theo cặp hai phần tử cuối)

```python
class Solution:
    def lenLongestFibSubseq(self, arr: List[int]) -> int:
        n = len(arr)
        d = {x: i for i, x in enumerate(arr)}         # giá trị -> chỉ số (arr tăng ngặt nên duy nhất)
        f = [[2] * n for _ in range(n)]                 # f[i][j]: dãy dài nhất kết thúc bằng arr[j], arr[i]
        ans = 0
        for i in range(2, n):
            for j in range(1, i):
                k = d.get(arr[i] - arr[j])              # phần tử đứng trước arr[j] trong dãy
                if k is not None and k < j:             # phải nằm TRƯỚC arr[j]
                    f[i][j] = f[j][k] + 1
                    ans = max(ans, f[i][j])
        return ans                                      # 0 nếu không có dãy độ dài >= 3
```

Công thức nhớ nhanh: `f[i][j] = f[j][k] + 1` với `arr[k] = arr[i] - arr[j]`, `k < j`; đáp án chỉ tính khi tìm được `k` (độ dài `>= 3`).

### Checklist trước khi submit

- [ ] Đáp án khởi tạo `0` (không phải `2`) và chỉ ghi nhận độ dài `>= 3`?
- [ ] Có điều kiện thứ tự `k < j` (hoặc `t < arr[j]`)?
- [ ] Thử **mọi** cặp, không chỉ cặp liền kề hay cặp đầu của mảng?
- [ ] Dùng bảng băm `giá trị -> chỉ số` (không `list.index`) để giữ `O(n^2)`?
- [ ] Đã thử: `[1,2,4]` (đáp án `0`), cả mảng là Fibonacci, Lucas, biên `10^9`, `n = 1000`?
- [ ] Tên method đúng `lenLongestFibSubseq(self, arr)` và trả `int`?